## Inspect OpenAI API Usage

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../data/gpt_interactions.csv")  # aggiorna se necessario
assert csv_path.exists(), f"File non trovato: {csv_path.resolve()}"

df = pd.read_csv(csv_path)

# Assicura che le colonne nuove esistano anche per CSV vecchi
expected_cols = [
    "api_prompt_tokens", "api_completion_tokens", "api_total_tokens", "api_cached_tokens",
    "image_count", "image_detail", "image_total_bytes", "image_est_tokens_low", "image_est_tokens_high",
]
for c in expected_cols:
    if c not in df.columns:
        df[c] = -1


In [2]:
import numpy as np

# Prezzi "list" OpenAI per 1M token (USD).
# Nota: per Azure sostituisci questi valori con il tuo listino.
PRICING_TABLE = {
    "gpt-4o":        {"input_per_1m": 2.50, "cached_input_per_1m": 1.25,  "output_per_1m": 10.00},
    "gpt-4o-mini":   {"input_per_1m": 0.15, "cached_input_per_1m": 0.075, "output_per_1m": 0.60},
    "gpt-4.1":       {"input_per_1m": 2.00, "cached_input_per_1m": 0.50,  "output_per_1m": 8.00},
    "gpt-4.1-mini":  {"input_per_1m": 0.40, "cached_input_per_1m": 0.10,  "output_per_1m": 1.60},
    "gpt-4.1-nano":  {"input_per_1m": 0.10, "cached_input_per_1m": 0.025, "output_per_1m": 0.40},
    "gpt-5":         {"input_per_1m": 1.25, "cached_input_per_1m": 0.125, "output_per_1m": 10.00},
    "gpt-5-mini":    {"input_per_1m": 0.25, "cached_input_per_1m": 0.025, "output_per_1m": 2.00},
    "gpt-5-nano":    {"input_per_1m": 0.05, "cached_input_per_1m": 0.005, "output_per_1m": 0.40},
    "gpt-5.2":       {"input_per_1m": 1.75, "cached_input_per_1m": 0.175, "output_per_1m": 14.00},
    "o4-mini":       {"input_per_1m": 1.10, "cached_input_per_1m": 0.275, "output_per_1m": 4.40},
    "o3":            {"input_per_1m": 2.00, "cached_input_per_1m": 0.50,  "output_per_1m": 8.00},
    "o3-mini":       {"input_per_1m": 1.10, "cached_input_per_1m": 0.55,  "output_per_1m": 4.40},
}

def resolve_pricing_key(model_name: str) -> str | None:
    """
    Mappa il model_name (spesso deployment Azure) a una chiave del PRICING_TABLE.
    Strategia: match per substring, preferendo la chiave più lunga (più specifica).
    """
    if not isinstance(model_name, str) or not model_name:
        return None
    m = model_name.lower()

    matches = [k for k in PRICING_TABLE.keys() if k in m]
    if not matches:
        return None
    # preferisci match più specifico
    matches.sort(key=len, reverse=True)
    return matches[0]

def _safe_int(x, default=-1):
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default

def _safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def compute_cost_row(row) -> dict:
    """
    Ritorna un dict con:
      - pricing_key
      - tokens_input_used / tokens_output_used (preferisce api_*; fallback)
      - cost_usd (miglior stima unica)
      - cost_usd_low / cost_usd_high (range quando manca api_prompt_tokens e ci sono immagini stimate)
      - note (spiega fallback usato)
    """
    model_name = row.get("model_name", "")
    pricing_key = resolve_pricing_key(model_name)
    if pricing_key is None:
        return {
            "pricing_key": None,
            "tokens_input_used": np.nan,
            "tokens_output_used": np.nan,
            "cost_usd": np.nan,
            "cost_usd_low": np.nan,
            "cost_usd_high": np.nan,
            "note": "no_pricing_match",
        }

    prices = PRICING_TABLE[pricing_key]
    input_rate = prices["input_per_1m"] / 1_000_000.0
    cached_rate = prices.get("cached_input_per_1m", prices["input_per_1m"]) / 1_000_000.0
    output_rate = prices["output_per_1m"] / 1_000_000.0

    # Preferisci usage API (include anche image tokens quando presenti nel prompt)
    api_prompt = _safe_int(row.get("api_prompt_tokens", -1), -1)
    api_comp = _safe_int(row.get("api_completion_tokens", -1), -1)
    api_cached = _safe_int(row.get("api_cached_tokens", -1), -1)

    # Fallback vecchie colonne
    text_prompt_est = _safe_int(row.get("token_count_prompt", 0), 0)
    text_comp_est = _safe_int(row.get("token_count_completion", 0), 0)

    # Stima immagini (se manca api_prompt_tokens)
    img_low = _safe_int(row.get("image_est_tokens_low", -1), -1)
    img_high = _safe_int(row.get("image_est_tokens_high", -1), -1)
    img_detail = str(row.get("image_detail", "") or "").lower()

    note = ""

    if api_prompt >= 0:
        # Uso reale (più preciso)
        prompt_tokens_used = api_prompt
        completion_tokens_used = api_comp if api_comp >= 0 else text_comp_est

        cached_tokens_used = max(0, min(api_cached, prompt_tokens_used)) if api_cached >= 0 else 0
        uncached_tokens_used = max(0, prompt_tokens_used - cached_tokens_used)

        cost = (cached_tokens_used * cached_rate) + (uncached_tokens_used * input_rate) + (completion_tokens_used * output_rate)

        return {
            "pricing_key": pricing_key,
            "tokens_input_used": prompt_tokens_used,
            "tokens_output_used": completion_tokens_used,
            "cost_usd": round(cost, 8),
            "cost_usd_low": round(cost, 8),
            "cost_usd_high": round(cost, 8),
            "note": "api_usage" + ("" if api_cached >= 0 else "_no_cached_info"),
        }

    # Se non ho api_prompt_tokens: uso testo + stima immagini se disponibile
    # Crea un range se detail=auto (o non specificato) e abbiamo low/high.
    prompt_low = text_prompt_est + (img_low if img_low >= 0 else 0)
    prompt_high = text_prompt_est + (img_high if img_high >= 0 else 0)

    completion_tokens_used = text_comp_est

    # best single estimate: se detail=low/high uso quello; se auto uso "high" (conservativo)
    if img_low >= 0 or img_high >= 0:
        if img_detail == "low":
            prompt_best = prompt_low
            note = "fallback_text_plus_image_low"
        elif img_detail == "high":
            prompt_best = prompt_high
            note = "fallback_text_plus_image_high"
        else:
            prompt_best = prompt_high
            note = "fallback_text_plus_image_auto_as_high"
    else:
        prompt_best = text_prompt_est
        prompt_low = text_prompt_est
        prompt_high = text_prompt_est
        note = "fallback_text_only"

    # cached info non disponibile in fallback: assumo 0 cached
    cost_low = (prompt_low * input_rate) + (completion_tokens_used * output_rate)
    cost_high = (prompt_high * input_rate) + (completion_tokens_used * output_rate)
    cost_best = (prompt_best * input_rate) + (completion_tokens_used * output_rate)

    return {
        "pricing_key": pricing_key,
        "tokens_input_used": prompt_best,
        "tokens_output_used": completion_tokens_used,
        "cost_usd": round(cost_best, 8),
        "cost_usd_low": round(cost_low, 8),
        "cost_usd_high": round(cost_high, 8),
        "note": note,
    }

# Applica calcolo
cost_cols = df.apply(compute_cost_row, axis=1, result_type="expand")
# Se la cella è stata eseguita più volte, rimuovi eventuali colonne già create
cols_to_drop_if_exist = [
    "pricing_key", "tokens_input_used", "tokens_output_used",
    "cost_usd", "cost_usd_low", "cost_usd_high", "note", "cost_eur_approx"
]
df = df.drop(columns=[c for c in cols_to_drop_if_exist if c in df.columns], errors="ignore")

cost_cols = df.apply(compute_cost_row, axis=1, result_type="expand")
df = pd.concat([df, cost_cols], axis=1)

EUR_PER_USD_APPROX = 0.90
df["cost_eur_approx"] = df["cost_usd"] * EUR_PER_USD_APPROX

# Colonna comoda: cost in EUR (approssimazione come facevi tu)
EUR_PER_USD_APPROX = 0.90
df["cost_eur_approx"] = df["cost_usd"] * EUR_PER_USD_APPROX


In [3]:
from agentz.utility import print_dataframe

cols_show = [
    "timestamp",
    "model_name",
    "pricing_key",
    "success",
    "api_prompt_tokens",
    "api_completion_tokens",
    "api_cached_tokens",
    "token_count_prompt",
    "token_count_completion",
    "image_count",
    "image_detail",
    "image_est_tokens_low",
    "image_est_tokens_high",
    "cost_usd",
    "cost_usd_low",
    "cost_usd_high",
    "note",
]

# Mostra ultime N righe per leggibilità
print_dataframe(df[cols_show].tail(50))

total_usd = df["cost_usd"].sum(skipna=True)
total_eur = df["cost_eur_approx"].sum(skipna=True)

# Se vuoi anche un range totale (utile quando molte righe sono fallback):
total_usd_low = df["cost_usd_low"].sum(skipna=True)
total_usd_high = df["cost_usd_high"].sum(skipna=True)

print(f"Spesa totale stimata: {total_usd:.6f}$ (~{total_eur:.6f}€) su {len(df)} test.")
print(f"Range (quando applicabile): {total_usd_low:.6f}$  -  {total_usd_high:.6f}$")

# Breakdown per modello (utile per capire dove spendi)
by_model = (
    df.groupby(["pricing_key"], dropna=False)[["cost_usd", "cost_usd_low", "cost_usd_high"]]
      .sum()
      .sort_values("cost_usd", ascending=False)
)
print_dataframe(by_model.reset_index())


┌──────┬────────────────────────────┬──────────────┬───────────────┬───────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬──────────────────────────┬───────────────┬────────────────┬────────────────────────┬─────────────────────────┬────────────┬────────────────┬─────────────────┬──────────────────────────┐
│      │ timestamp                  │ model_name   │ pricing_key   │ success   │   api_prompt_tokens │   api_completion_tokens │   api_cached_tokens │   token_count_prompt │   token_count_completion │   image_count │ image_detail   │   image_est_tokens_low │   image_est_tokens_high │   cost_usd │   cost_usd_low │   cost_usd_high │ note                     │
├──────┼────────────────────────────┼──────────────┼───────────────┼───────────┼─────────────────────┼─────────────────────────┼─────────────────────┼──────────────────────┼──────────────────────────┼───────────────┼────────────────┼────────────────────────┼──────────────────────